# Decision Tree สำหรับทำนายความเสี่ยงการเสียชีวิตของผู้ป่วย COVID-19

Notebook นี้สร้างและประเมินโมเดล **Decision Tree** เพื่อทำนาย `DEATH_STATUS`

- `0` = รอดชีวิต (Survived)
- `1` = เสียชีวิต (Died)

จุดประสงค์คือพัฒนาโมเดลของสมาชิกคนที่ 1 และบันทึกผลในรูปแบบเดียวกับ SVM เพื่อใช้สร้าง `model_comparison.ipynb` ภายหลัง

> โปรเจกต์นี้จัดทำเพื่อการศึกษา ไม่ใช่เครื่องมือวินิจฉัยหรือใช้ตัดสินใจทางการแพทย์

## ขั้นตอนการทดลอง

1. ตรวจสอบข้อมูลและคุณภาพข้อมูล
2. สำรวจสัดส่วน Target, Missing values, Duplicate rows และอายุ
3. ใช้ preprocessing และ train/test split ส่วนกลาง
4. ปรับพารามิเตอร์ด้วย Stratified 5-Fold Cross-Validation บน training set
5. เทรนโมเดลที่ดีที่สุดและประเมิน test set เพียงครั้งเดียว
6. แสดง Confusion Matrix, ROC Curve, Precision-Recall Curve, ต้นไม้ และ Feature Importance
7. บันทึก metrics และกราฟใน `results/`

เกณฑ์หลักในการเลือกพารามิเตอร์คือ **F1-score ของคลาส Died** เพราะข้อมูลมีคลาสไม่สมดุล

In [ ]:
from pathlib import Path
import sys
import time
import warnings

# รองรับการรันจากโฟลเดอร์หลักและจากโฟลเดอร์ notebooks
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ใช้เฉพาะกรณี Python หลักไม่มี packaging แต่ .venv มีอยู่
try:
    import packaging  # noqa: F401
except ModuleNotFoundError:
    venv_site_packages = PROJECT_ROOT / ".venv" / "Lib" / "site-packages"
    if venv_site_packages.exists():
        sys.path.append(str(venv_site_packages))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import PercentFormatter
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

from src.evaluate import evaluate_classifier, save_metrics
from src.preprocess import (
    CATEGORICAL_FEATURES,
    DATA_PATH,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    build_preprocessor,
    get_train_test_data,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

# เลือกฟอนต์ที่รองรับภาษาไทย เพื่อให้ข้อความในกราฟแสดงผลถูกต้อง
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
thai_font_candidates = ["Leelawadee UI", "Tahoma", "Nirmala UI", "Arial Unicode MS", "DejaVu Sans"]
THAI_FONT = next((font for font in thai_font_candidates if font in available_fonts), "DejaVu Sans")
plt.rcParams["font.family"] = THAI_FONT
plt.rcParams["axes.unicode_minus"] = False

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures" / "decision_tree"
METRICS_PATH = RESULTS_DIR / "decision_tree_metrics.json"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS_TH = {0: "รอดชีวิต", 1: "เสียชีวิต", "Survived": "รอดชีวิต", "Died": "เสียชีวิต"}
FEATURE_LABELS_TH = {
    "AGE": "อายุ",
    "USMER": "หน่วยบริการ USMER",
    "MEDICAL_UNIT": "หน่วยงานทางการแพทย์",
    "SEX": "เพศ",
    "PATIENT_TYPE": "ประเภทผู้ป่วย",
    "PNEUMONIA": "ภาวะปอดอักเสบ",
    "PREGNANT": "การตั้งครรภ์",
    "DIABETES": "โรคเบาหวาน",
    "COPD": "โรคปอดอุดกั้นเรื้อรัง",
    "ASTHMA": "โรคหอบหืด",
    "INMSUPR": "ภาวะภูมิคุ้มกันบกพร่อง",
    "HIPERTENSION": "โรคความดันโลหิตสูง",
    "OTHER_DISEASE": "โรคประจำตัวอื่น",
    "CARDIOVASCULAR": "โรคหัวใจและหลอดเลือด",
    "OBESITY": "โรคอ้วน",
    "RENAL_CHRONIC": "โรคไตเรื้อรัง",
    "TOBACCO": "การสูบบุหรี่",
    "CLASIFFICATION_FINAL": "ผลจำแนก COVID-19",
}
METRIC_LABELS_TH = {
    "accuracy": "Accuracy (ความถูกต้องรวม)",
    "precision": "Precision (ความแม่นยำเมื่อทำนายว่าเสียชีวิต)",
    "recall": "Recall (สัดส่วนผู้เสียชีวิตที่ตรวจพบ)",
    "f1_score": "F1-score (สมดุลระหว่าง Precision และ Recall)",
    "roc_auc": "ROC-AUC (ความสามารถในการแยกสองคลาส)",
    "pr_auc": "PR-AUC (คุณภาพบนคลาสเสียชีวิตที่มีจำนวนน้อย)",
}

def translate_feature_name(feature_name):
    clean_name = feature_name.replace("numeric__", "").replace("categorical__", "")
    for raw_name in sorted(FEATURE_LABELS_TH, key=len, reverse=True):
        if clean_name == raw_name:
            return f"{FEATURE_LABELS_TH[raw_name]} ({raw_name})"
        prefix = f"{raw_name}_"
        if clean_name.startswith(prefix):
            raw_code = clean_name[len(prefix):]
            try:
                code = int(float(raw_code))
            except ValueError:
                code = raw_code

            if raw_name == "SEX":
                code_label = {1: "หญิง", 2: "ชาย"}.get(code, f"รหัส {code}")
            elif raw_name == "PATIENT_TYPE":
                code_label = {1: "ผู้ป่วยนอก", 2: "ผู้ป่วยใน"}.get(code, f"รหัส {code}")
            elif raw_name == "PREGNANT":
                code_label = {1: "ตั้งครรภ์", 2: "ไม่ตั้งครรภ์"}.get(code, f"รหัส {code}")
            elif raw_name in {"PNEUMONIA", "DIABETES", "COPD", "ASTHMA", "INMSUPR", "HIPERTENSION", "OTHER_DISEASE", "CARDIOVASCULAR", "OBESITY", "RENAL_CHRONIC", "TOBACCO"}:
                code_label = {1: "มี", 2: "ไม่มี"}.get(code, f"รหัส {code}")
            else:
                code_label = f"รหัส {code}"
            return f"{FEATURE_LABELS_TH[raw_name]}: {code_label}"
    return clean_name

def save_figure(fig, filename):
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    print(f"บันทึกกราฟแล้ว: {output_path.relative_to(PROJECT_ROOT)}")

print(f"ฟอนต์ที่ใช้ในกราฟ: {THAI_FONT}")
print(f"โฟลเดอร์โปรเจกต์: {PROJECT_ROOT}")
print(f"ชุดข้อมูล: {DATA_PATH}")

## 1. ตรวจสอบข้อมูลดิบ

ตารางนี้ใช้ตรวจจำนวนแถว คอลัมน์ Missing values และข้อมูลซ้ำก่อน preprocessing

**การตัดสินใจเรื่อง Duplicate:** Notebook จะแสดงจำนวนข้อมูลซ้ำแต่ยังไม่ลบทิ้ง เพราะ Dataset ไม่มีรหัสผู้ป่วยที่ใช้ยืนยันว่าเป็นบุคคลเดียวกัน การลบอาจทำให้ผู้ป่วยคนละคนที่มีคุณลักษณะเหมือนกันหายไป ทั้ง Decision Tree และ SVM จึงใช้ข้อมูลชุดเดียวกันตาม `src/preprocess.py`

In [ ]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)

dataset_audit = pd.DataFrame(
    {
        "รายการ": [
            "จำนวนแถว",
            "จำนวนคอลัมน์ทั้งหมด",
            "จำนวน features ที่ใช้",
            "จำนวนแถวซ้ำทั้งแถว",
            "Target ที่หาย",
        ],
        "ค่า": [
            len(raw_df),
            raw_df.shape[1],
            len(FEATURE_COLUMNS),
            int(raw_df.duplicated().sum()),
            int(raw_df[TARGET_COLUMN].isna().sum()),
        ],
    }
)

display(dataset_audit)

preview_columns_th = {
    **{column: f"{FEATURE_LABELS_TH.get(column, column)} ({column})" for column in FEATURE_COLUMNS},
    TARGET_COLUMN: "สถานะการเสียชีวิต (DEATH_STATUS)",
}
raw_preview = raw_df[FEATURE_COLUMNS + [TARGET_COLUMN]].head().rename(columns=preview_columns_th)
raw_preview["สถานะการเสียชีวิต (DEATH_STATUS)"] = raw_preview["สถานะการเสียชีวิต (DEATH_STATUS)"].map(CLASS_LABELS_TH)
display(raw_preview)

## 2. การกระจายของ Target

หากทำนายทุกคนเป็น `Survived` จะได้ Accuracy สูงเพราะข้อมูลไม่สมดุล แต่ Recall และ F1 ของ `Died` จะเป็นศูนย์ จึงต้องใช้ Precision, Recall, F1-score และ PR-AUC ร่วมกัน

In [ ]:
target_summary = (
    raw_df[TARGET_COLUMN]
    .value_counts()
    .rename_axis("ผลลัพธ์")
    .reset_index(name="จำนวนผู้ป่วย")
)
target_summary["ผลลัพธ์"] = target_summary["ผลลัพธ์"].map(CLASS_LABELS_TH)
target_summary["ร้อยละ"] = target_summary["จำนวนผู้ป่วย"] / len(raw_df) * 100
display(target_summary.style.format({"จำนวนผู้ป่วย": "{:,.0f}", "ร้อยละ": "{:.2f}%"}))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(
    data=target_summary,
    x="ผลลัพธ์",
    y="จำนวนผู้ป่วย",
    hue="ผลลัพธ์",
    palette={"รอดชีวิต": "#4C78A8", "เสียชีวิต": "#E45756"},
    legend=False,
    ax=ax,
)
ax.set_title("สัดส่วนผลลัพธ์ของผู้ป่วยในชุดข้อมูล")
ax.set_xlabel("")
ax.set_ylabel("จำนวนผู้ป่วย (คน)")
for bar, (_, row) in zip(ax.patches, target_summary.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{row['จำนวนผู้ป่วย']:,.0f}\n({row['ร้อยละ']:.2f}%)",
        ha="center",
        va="bottom",
    )
ax.set_ylim(0, target_summary["จำนวนผู้ป่วย"].max() * 1.12)
fig.tight_layout()
save_figure(fig, "01_target_distribution.png")
plt.show()

## 3. Missing values และรหัส 97, 98, 99

ใน Dataset นี้ค่า `97`, `98` และ `99` ของตัวแปรหมวดหมู่มักหมายถึงไม่ทราบหรือไม่เกี่ยวข้อง จึงแปลงเป็น Missing ก่อนแสดงรายงาน เช่น `PREGNANT` อาจเป็นไม่เกี่ยวข้องสำหรับผู้ป่วยชาย

โมเดลใช้ `SimpleImputer(strategy="most_frequent")` จาก preprocessing ส่วนกลาง เพื่อให้ทั้งสองโมเดลอยู่ภายใต้เงื่อนไขเดียวกัน

In [ ]:
audit_features = raw_df[FEATURE_COLUMNS].copy()

for column in FEATURE_COLUMNS:
    audit_features[column] = pd.to_numeric(
        audit_features[column], errors="coerce"
    )

for column in CATEGORICAL_FEATURES:
    audit_features[column] = audit_features[column].replace(
        [97, 98, 99], np.nan
    )

missing_summary = pd.DataFrame(
    {
        "ตัวแปร": [f"{FEATURE_LABELS_TH.get(c, c)} ({c})" for c in FEATURE_COLUMNS],
        "จำนวนค่าที่หาย": [audit_features[c].isna().sum() for c in FEATURE_COLUMNS],
    }
)
missing_summary["ร้อยละที่หาย"] = (
    missing_summary["จำนวนค่าที่หาย"] / len(audit_features) * 100
)
missing_summary = missing_summary.sort_values(
    "ร้อยละที่หาย", ascending=False
).reset_index(drop=True)

display(missing_summary.style.format({"จำนวนค่าที่หาย": "{:,.0f}", "ร้อยละที่หาย": "{:.2f}%"}))

missing_for_plot = missing_summary[missing_summary["จำนวนค่าที่หาย"] > 0]
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(
    data=missing_for_plot,
    x="ร้อยละที่หาย",
    y="ตัวแปร",
    color="#F2CF5B",
    ax=ax,
)
ax.set_title("สัดส่วนข้อมูลที่หาย หลังแปลงรหัส 97, 98 และ 99")
ax.set_xlabel("ข้อมูลที่หาย (%)")
ax.set_ylabel("")
fig.tight_layout()
save_figure(fig, "02_missing_values.png")
plt.show()

## 4. อายุและความเสี่ยงการเสียชีวิต

ส่วนนี้แสดงการกระจายอายุแยกตามผลลัพธ์ และคำนวณอัตราการเสียชีวิตตามช่วงอายุ การวิเคราะห์นี้เป็นความสัมพันธ์ในข้อมูล ไม่ใช่ข้อสรุปเชิงสาเหตุทางการแพทย์

In [ ]:
age_summary = (
    raw_df.groupby(TARGET_COLUMN)["AGE"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)
age_summary = age_summary.rename(
    index=CLASS_LABELS_TH,
    columns={"count": "จำนวนผู้ป่วย", "mean": "อายุเฉลี่ย", "median": "มัธยฐานอายุ", "min": "อายุต่ำสุด", "max": "อายุสูงสุด"},
)
age_summary.index.name = "ผลลัพธ์"
display(age_summary)

age_bins = [0, 17, 29, 39, 49, 59, 69, 79, 200]
age_labels = ["0-17", "18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]

age_analysis = raw_df[["AGE", TARGET_COLUMN]].copy()
age_analysis["AGE_GROUP"] = pd.cut(
    age_analysis["AGE"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
)
age_analysis["DIED"] = (age_analysis[TARGET_COLUMN] == "Died").astype(int)

age_group_summary = (
    age_analysis.groupby("AGE_GROUP", observed=False)
    .agg(**{"จำนวนผู้ป่วย": ("DIED", "size"), "จำนวนผู้เสียชีวิต": ("DIED", "sum"), "อัตราการเสียชีวิต": ("DIED", "mean")})
    .reset_index()
)
age_group_summary = age_group_summary.rename(columns={"AGE_GROUP": "ช่วงอายุ (ปี)"})
age_group_summary["อัตราการเสียชีวิต"] *= 100
display(age_group_summary.style.format({"จำนวนผู้ป่วย": "{:,.0f}", "จำนวนผู้เสียชีวิต": "{:,.0f}", "อัตราการเสียชีวิต": "{:.2f}%"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
age_plot_data = raw_df.copy()
age_plot_data["ผลลัพธ์"] = age_plot_data[TARGET_COLUMN].map(CLASS_LABELS_TH)
sns.histplot(
    data=age_plot_data,
    x="AGE",
    hue="ผลลัพธ์",
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    palette={"รอดชีวิต": "#4C78A8", "เสียชีวิต": "#E45756"},
    ax=axes[0],
)
axes[0].set_title("การกระจายอายุ แยกตามผลลัพธ์")
axes[0].set_xlabel("อายุ (ปี)")
axes[0].set_ylabel("ความหนาแน่นของข้อมูล")

sns.barplot(
    data=age_group_summary,
    x="ช่วงอายุ (ปี)",
    y="อัตราการเสียชีวิต",
    color="#E45756",
    ax=axes[1],
)
axes[1].set_title("อัตราการเสียชีวิตตามช่วงอายุ")
axes[1].set_xlabel("ช่วงอายุ (ปี)")
axes[1].set_ylabel("อัตราการเสียชีวิต (%)")
axes[1].tick_params(axis="x", rotation=35)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f%%", padding=3)
axes[1].set_ylim(0, age_group_summary["อัตราการเสียชีวิต"].max() * 1.12)

fig.tight_layout()
save_figure(fig, "03_age_analysis.png")
plt.show()

## 5. โหลด Train/Test จากส่วนกลาง

`get_train_test_data()` กำหนดเงื่อนไขร่วมกันไว้ดังนี้:

- `test_size=0.20`
- `random_state=42`
- `stratify=y`
- Features 18 ตัว
- ไม่ใช้คอลัมน์ที่ทำให้เกิด Data Leakage เช่น `DATE_DIED`, `DEATH_YEAR`, `RECOVERY_STATUS` และ `RISK_CATEGORY`

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_data()

split_summary = pd.DataFrame(
    {
        "ชุดข้อมูล": ["ชุดฝึกสอน (Train)", "ชุดทดสอบ (Test)"],
        "จำนวนแถว": [len(X_train), len(X_test)],
        "รอดชีวิต": [(y_train == 0).sum(), (y_test == 0).sum()],
        "เสียชีวิต": [(y_train == 1).sum(), (y_test == 1).sum()],
        "สัดส่วนเสียชีวิต": [y_train.mean() * 100, y_test.mean() * 100],
    }
)
display(split_summary.style.format({"จำนวนแถว": "{:,.0f}", "รอดชีวิต": "{:,.0f}", "เสียชีวิต": "{:,.0f}", "สัดส่วนเสียชีวิต": "{:.2f}%"}))
print(f"ขนาดชุดฝึกสอน: {X_train.shape}")
print(f"ขนาดชุดทดสอบ: {X_test.shape}")

## 6. สร้าง Pipeline และปรับพารามิเตอร์

Pipeline ทำให้ preprocessing ถูก `fit` จาก training fold เท่านั้น จึงป้องกัน Data Leakage ระหว่าง Cross-Validation

พารามิเตอร์ที่ทดลอง:

- `criterion`: วิธีวัดคุณภาพของจุดแบ่ง
- `max_depth`: จำกัดความลึกเพื่อควบคุม Overfitting
- `min_samples_split`: จำนวนตัวอย่างขั้นต่ำก่อนแบ่งโหนด
- `min_samples_leaf`: จำนวนตัวอย่างขั้นต่ำใน Leaf
- `ccp_alpha`: Cost Complexity Pruning

In [ ]:
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "classifier",
            DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

parameter_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [3, 5, 7, 10],
    "classifier__min_samples_split": [2, 10],
    "classifier__min_samples_leaf": [5, 10, 20],
    "classifier__ccp_alpha": [0.0, 0.001],
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}

grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="f1",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True,
    verbose=0,
)

search_start = time.perf_counter()
grid_search.fit(X_train, y_train)
search_seconds = time.perf_counter() - search_start

print(f"ค้นหาพารามิเตอร์เสร็จใน {search_seconds:.2f} วินาที")
print(f"ค่า F1 เฉลี่ยจาก Cross-Validation ที่ดีที่สุด: {grid_search.best_score_:.4f}")
print("พารามิเตอร์ที่ดีที่สุด:")
parameter_names_th = {"criterion": "เกณฑ์แบ่งโหนด", "max_depth": "ความลึกสูงสุด", "min_samples_split": "ตัวอย่างขั้นต่ำก่อนแบ่ง", "min_samples_leaf": "ตัวอย่างขั้นต่ำในใบ", "ccp_alpha": "ค่าตัดแต่งต้นไม้"}
for name, value in grid_search.best_params_.items():
    short_name = name.replace("classifier__", "")
    print(f"  {parameter_names_th.get(short_name, short_name)} ({short_name}): {value}")

## 7. ผล Cross-Validation

ตารางแสดง 10 ชุดพารามิเตอร์ที่มีค่า Mean CV F1 สูงที่สุด ส่วนกราฟแสดงคะแนนที่ดีที่สุดในแต่ละความลึก

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

cv_columns = [
    "param_classifier__criterion",
    "param_classifier__max_depth",
    "param_classifier__min_samples_split",
    "param_classifier__min_samples_leaf",
    "param_classifier__ccp_alpha",
    "mean_train_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_recall",
    "mean_test_precision",
    "mean_test_pr_auc",
    "mean_test_roc_auc",
]

top_cv_results = (
    cv_results[cv_columns]
    .sort_values("mean_test_f1", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
cv_column_labels_th = {
    "param_classifier__criterion": "เกณฑ์แบ่งโหนด",
    "param_classifier__max_depth": "ความลึกสูงสุด",
    "param_classifier__min_samples_split": "ตัวอย่างขั้นต่ำก่อนแบ่ง",
    "param_classifier__min_samples_leaf": "ตัวอย่างขั้นต่ำในใบ",
    "param_classifier__ccp_alpha": "ค่าตัดแต่งต้นไม้",
    "mean_train_f1": "F1 ชุดฝึกเฉลี่ย",
    "mean_test_f1": "F1 ชุดตรวจสอบเฉลี่ย",
    "std_test_f1": "ส่วนเบี่ยงเบน F1",
    "mean_test_recall": "Recall เฉลี่ย",
    "mean_test_precision": "Precision เฉลี่ย",
    "mean_test_pr_auc": "PR-AUC เฉลี่ย",
    "mean_test_roc_auc": "ROC-AUC เฉลี่ย",
}
display(top_cv_results.rename(columns=cv_column_labels_th).style.format(precision=4))

depth_summary = (
    cv_results.groupby("param_classifier__max_depth", as_index=False)
    .agg(Best_CV_F1=("mean_test_f1", "max"))
    .sort_values("param_classifier__max_depth")
)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.lineplot(
    data=depth_summary,
    x="param_classifier__max_depth",
    y="Best_CV_F1",
    marker="o",
    linewidth=2.5,
    color="#4C78A8",
    ax=ax,
)
ax.set_title("ค่า F1 ที่ดีที่สุดในแต่ละระดับความลึกของต้นไม้")
ax.set_xlabel("ความลึกสูงสุดของต้นไม้ (max_depth)")
ax.set_ylabel("ค่า F1 เฉลี่ยจาก Cross-Validation")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
for row in depth_summary.itertuples(index=False):
    ax.annotate(f"{row.Best_CV_F1:.1%}", (row.param_classifier__max_depth, row.Best_CV_F1), textcoords="offset points", xytext=(0, 8), ha="center")
ax.set_ylim(0, depth_summary["Best_CV_F1"].max() * 1.15)
ax.set_xticks(depth_summary["param_classifier__max_depth"])
fig.tight_layout()
save_figure(fig, "04_cv_f1_by_depth.png")
plt.show()

## 8. ประเมินโมเดลที่ดีที่สุดบน Test Set

Test set ถูกใช้หลังจากเลือกพารามิเตอร์เสร็จแล้วเท่านั้น นอกจากนี้ยังสร้าง Dummy Baseline ที่ทำนายทุกคนเป็น `Survived` เพื่อแสดงว่า Accuracy สูงเพียงอย่างเดียวไม่เพียงพอ

In [ ]:
best_decision_tree = grid_search.best_estimator_

train_pred = best_decision_tree.predict(X_train)
train_score = best_decision_tree.predict_proba(X_train)[:, 1]
test_pred = best_decision_tree.predict(X_test)
test_score = best_decision_tree.predict_proba(X_test)[:, 1]

train_metrics = evaluate_classifier(y_train, train_pred, train_score)
test_metrics = evaluate_classifier(y_test, test_pred, test_score)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
dummy_pred = dummy.predict(np.zeros((len(y_test), 1)))
dummy_score = np.zeros(len(y_test))
dummy_metrics = evaluate_classifier(y_test, dummy_pred, dummy_score)

metric_names = ["accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]
comparison_table = pd.DataFrame(
    {
        "ตัวชี้วัดและความหมาย": [METRIC_LABELS_TH[m] for m in metric_names],
        "Decision Tree": [test_metrics[m] for m in metric_names],
        "โมเดลพื้นฐาน (ทายว่ารอดชีวิตทั้งหมด)": [dummy_metrics[m] for m in metric_names],
    }
)
display(comparison_table.style.format({"Decision Tree": "{:.2%}", "โมเดลพื้นฐาน (ทายว่ารอดชีวิตทั้งหมด)": "{:.2%}"}))

true_negative, false_positive, false_negative, true_positive = np.asarray(test_metrics["confusion_matrix"]).ravel()
prediction_summary = pd.DataFrame(
    {
        "ผลการทำนาย": [
            "ทำนายผู้รอดชีวิตถูกต้อง (True Negative)",
            "ทำนายว่าเสียชีวิตเกินจริง (False Positive)",
            "พลาดผู้เสียชีวิตจริง (False Negative)",
            "ตรวจพบผู้เสียชีวิตถูกต้อง (True Positive)",
        ],
        "จำนวนผู้ป่วย (คน)": [true_negative, false_positive, false_negative, true_positive],
        "คำอธิบายสำหรับพรีเซนต์": [
            "ผู้ป่วยรอดชีวิตจริง และโมเดลทำนายว่ารอดชีวิต",
            "ผู้ป่วยรอดชีวิตจริง แต่โมเดลแจ้งว่าเสี่ยงเสียชีวิต",
            "ผู้ป่วยเสียชีวิตจริง แต่โมเดลทำนายว่ารอดชีวิต",
            "ผู้ป่วยเสียชีวิตจริง และโมเดลตรวจพบได้ถูกต้อง",
        ],
    }
)
display(prediction_summary.style.format({"จำนวนผู้ป่วย (คน)": "{:,.0f}"}))

print("สรุปสำหรับพรีเซนต์")
print(f"- โมเดลตรวจพบผู้เสียชีวิตได้ {true_positive:,.0f} จาก {(true_positive + false_negative):,.0f} คน หรือ Recall {test_metrics['recall']:.2%}")
print(f"- เมื่อโมเดลทำนายว่าเสียชีวิต มีความแม่นยำ Precision {test_metrics['precision']:.2%}")
print(f"- F1-score ของคลาสเสียชีวิตเท่ากับ {test_metrics['f1_score']:.2%}")
print(f"- Accuracy ของ Decision Tree เท่ากับ {test_metrics['accuracy']:.2%} ส่วนโมเดลพื้นฐานเท่ากับ {dummy_metrics['accuracy']:.2%}")
print("- โมเดลพื้นฐานมี Accuracy สูงกว่าเพราะทายว่าทุกคนรอดชีวิต แต่ตรวจพบผู้เสียชีวิตไม่ได้เลย")

## 9. ตรวจ Overfitting จาก Train/Test Gap

หากคะแนน Train สูงกว่า Test มาก แสดงว่าโมเดลอาจ Overfit แม้จะใช้ Pruning แล้วก็ตาม

In [ ]:
generalization_table = pd.DataFrame(
    {
        "ตัวชี้วัด": ["Accuracy (ความถูกต้องรวม)", "Precision (ความแม่นยำ)", "Recall (การตรวจพบ)", "F1-score", "ROC-AUC", "PR-AUC"],
        "ชุดฝึกสอน (Train)": [
            train_metrics["accuracy"],
            train_metrics["precision"],
            train_metrics["recall"],
            train_metrics["f1_score"],
            train_metrics["roc_auc"],
            train_metrics["pr_auc"],
        ],
        "ชุดทดสอบ (Test)": [
            test_metrics["accuracy"],
            test_metrics["precision"],
            test_metrics["recall"],
            test_metrics["f1_score"],
            test_metrics["roc_auc"],
            test_metrics["pr_auc"],
        ],
    }
)
generalization_table["ผลต่าง Train-Test"] = generalization_table["ชุดฝึกสอน (Train)"] - generalization_table["ชุดทดสอบ (Test)"]
display(generalization_table.style.format({"ชุดฝึกสอน (Train)": "{:.2%}", "ชุดทดสอบ (Test)": "{:.2%}", "ผลต่าง Train-Test": "{:+.2%}"}))

classification_report_table = pd.DataFrame(test_metrics["classification_report"]).T
classification_report_table = classification_report_table.rename(
    index={"Survived": "รอดชีวิต", "Died": "เสียชีวิต", "accuracy": "ความถูกต้องรวม", "macro avg": "ค่าเฉลี่ยทุกคลาส", "weighted avg": "ค่าเฉลี่ยถ่วงน้ำหนัก"},
    columns={"precision": "Precision (ความแม่นยำ)", "recall": "Recall (การตรวจพบ)", "f1-score": "F1-score", "support": "จำนวนตัวอย่าง"},
)
classification_report_table.loc["ความถูกต้องรวม", "จำนวนตัวอย่าง"] = np.nan
display(classification_report_table.style.format({"Precision (ความแม่นยำ)": "{:.2%}", "Recall (การตรวจพบ)": "{:.2%}", "F1-score": "{:.2%}", "จำนวนตัวอย่าง": "{:,.0f}"}))

## 10. Confusion Matrix (ตารางผลการทำนาย)

วิธีอ่านตาราง:

- **True Positive:** เสียชีวิตจริง และโมเดลทำนายว่าเสียชีวิต
- **True Negative:** รอดชีวิตจริง และโมเดลทำนายว่ารอดชีวิต
- **False Positive:** รอดชีวิตจริง แต่โมเดลทำนายว่าเสียชีวิต
- **False Negative:** เสียชีวิตจริง แต่โมเดลทำนายว่ารอดชีวิต

ให้พิจารณา **False Negative** เป็นพิเศษ เพราะเป็นผู้เสียชีวิตจริงที่โมเดลตรวจไม่พบ

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    labels=[0, 1],
    display_labels=["รอดชีวิต", "เสียชีวิต"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("ตารางผลการทำนายของโมเดล Decision Tree")
ax.set_xlabel("ผลลัพธ์ที่โมเดลทำนาย")
ax.set_ylabel("ผลลัพธ์จริง")
fig.tight_layout()
save_figure(fig, "05_confusion_matrix.png")
plt.show()

## 11. ROC Curve และ Precision-Recall Curve

- **ROC Curve:** แสดงความสามารถของโมเดลในการแยกผู้รอดชีวิตกับผู้เสียชีวิต ค่า ROC-AUC ใกล้ 1 หมายถึงแยกได้ดี
- **Precision-Recall Curve:** แสดงสมดุลระหว่างความแม่นยำเมื่อแจ้งว่าเสียชีวิต กับสัดส่วนผู้เสียชีวิตที่ตรวจพบ
- เส้นของโมเดลพื้นฐานใช้เปรียบเทียบกับการทำนายแบบไม่มีความสามารถในการจำแนก

ข้อมูลนี้มีผู้เสียชีวิตจำนวนน้อย จึงควรให้ความสำคัญกับ **Precision-Recall Curve และ PR-AUC** มากเป็นพิเศษ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

roc_tree = RocCurveDisplay.from_predictions(
    y_test,
    test_score,
    name="Decision Tree",
    ax=axes[0],
)
roc_tree.line_.set(color="#4C78A8")

roc_dummy = RocCurveDisplay.from_predictions(
    y_test,
    dummy_score,
    name="โมเดลพื้นฐาน",
    ax=axes[0],
)
roc_dummy.line_.set(color="#999999", linestyle="--")
axes[0].legend()
axes[0].set_title("กราฟ ROC: ความสามารถในการแยกสองคลาส")
axes[0].set_xlabel("อัตราการแจ้งเตือนผิด (False Positive Rate)")
axes[0].set_ylabel("อัตราการตรวจพบผู้เสียชีวิต (True Positive Rate)")

pr_tree = PrecisionRecallDisplay.from_predictions(
    y_test,
    test_score,
    name="Decision Tree",
    ax=axes[1],
)
pr_tree.line_.set(color="#E45756")

pr_dummy = PrecisionRecallDisplay.from_predictions(
    y_test,
    dummy_score,
    name="โมเดลพื้นฐาน",
    ax=axes[1],
)
pr_dummy.line_.set(color="#999999", linestyle="--")
axes[1].legend()
axes[1].set_title("กราฟ Precision-Recall: เน้นคลาสเสียชีวิต")
axes[1].set_xlabel("Recall: สัดส่วนผู้เสียชีวิตที่ตรวจพบ")
axes[1].set_ylabel("Precision: ความแม่นยำเมื่อทำนายว่าเสียชีวิต")

fig.tight_layout()
save_figure(fig, "06_roc_pr_curves.png")
plt.show()

## 12. แสดงต้นไม้การตัดสินใจ

แสดงเพียง 3 ระดับแรกเพื่อให้อ่านง่าย ภาพนี้ไม่ใช่โครงสร้างทั้งหมดของโมเดล

คำแปลข้อความภายในแต่ละโหนด:

- **entropy:** ระดับความปะปนของสองคลาส ค่ายิ่งต่ำยิ่งแยกคลาสได้ชัด
- **samples:** สัดส่วนข้อมูลฝึกที่มาถึงโหนดนั้น
- **value:** สัดส่วนของคลาสรอดชีวิตและเสียชีวิตหลังคำนึงถึงน้ำหนักคลาส
- **class:** ผลลัพธ์ที่โหนดนั้นเลือกทำนาย
- กิ่งซ้ายหมายถึงเงื่อนไขเป็นจริง ส่วนกิ่งขวาหมายถึงเงื่อนไขเป็นเท็จ

> ค่าเกณฑ์ของ `AGE` ในต้นไม้เป็นค่าหลัง StandardScaler ไม่ใช่อายุจริงเป็นปี

In [ ]:
fitted_preprocessor = best_decision_tree.named_steps["preprocessor"]
fitted_classifier = best_decision_tree.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()

readable_feature_names = [translate_feature_name(name) for name in feature_names]

fig, ax = plt.subplots(figsize=(24, 11))
plot_tree(
    fitted_classifier,
    feature_names=readable_feature_names,
    class_names=["รอดชีวิต", "เสียชีวิต"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
    proportion=True,
    ax=ax,
)
ax.set_title("ต้นไม้ตัดสินใจ 3 ระดับแรก", fontsize=16)
fig.tight_layout()
save_figure(fig, "07_decision_tree_first_3_levels.png")
plt.show()

print("กฎการตัดสินใจ 3 ระดับแรก:")
print(
    export_text(
        fitted_classifier,
        feature_names=readable_feature_names,
        max_depth=3,
    )
)

## 13. Feature Importance (ความสำคัญของตัวแปร)

กราฟเรียงตัวแปรที่โมเดลใช้แบ่งข้อมูลจากมากไปน้อย ค่าสูงหมายถึงตัวแปรนั้นมีบทบาทต่อกฎของต้นไม้มาก

> Feature Importance อธิบายการทำงานของโมเดลเท่านั้น ไม่ได้ยืนยันว่าตัวแปรนั้นเป็นสาเหตุของการเสียชีวิต

In [ ]:
feature_importance = (
    pd.DataFrame(
        {
            "ตัวแปร": readable_feature_names,
            "ความสำคัญ": fitted_classifier.feature_importances_,
        }
    )
    .sort_values("ความสำคัญ", ascending=False)
    .reset_index(drop=True)
)

top_features = feature_importance.head(15)
display(top_features.style.format({"ความสำคัญ": "{:.2%}"}))

fig, ax = plt.subplots(figsize=(12, 6.5))
sns.barplot(
    data=top_features,
    x="ความสำคัญ",
    y="ตัวแปร",
    color="#59A14F",
    ax=ax,
)
ax.set_title("15 ตัวแปรที่มีอิทธิพลต่อการแบ่งโหนดมากที่สุด")
ax.set_xlabel("สัดส่วนความสำคัญของตัวแปร")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("")
for container in ax.containers:
    labels = [f"{value:.1%}" for value in container.datavalues]
    ax.bar_label(container, labels=labels, padding=3)
ax.set_xlim(0, top_features["ความสำคัญ"].max() * 1.10)
fig.tight_layout()
save_figure(fig, "08_feature_importance.png")
plt.show()

## 14. บันทึกผลการทดลอง

ไฟล์ `results/decision_tree_metrics.json` เก็บผลที่ `model_comparison.ipynb` จะนำไปเปรียบเทียบกับ SVM ภายหลัง

In [ ]:
best_index = grid_search.best_index_
best_params = {
    name.replace("classifier__", ""): value
    for name, value in grid_search.best_params_.items()
}

final_metrics = dict(test_metrics)
final_metrics.update(
    {
        "model_name": "Decision Tree",
        "selection_metric": "F1-score of Died class",
        "best_params": best_params,
        "cv_best_f1": float(grid_search.best_score_),
        "cv_best_pr_auc": float(grid_search.cv_results_["mean_test_pr_auc"][best_index]),
        "cv_best_recall": float(grid_search.cv_results_["mean_test_recall"][best_index]),
        "grid_search_seconds": float(search_seconds),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "input_feature_count": int(len(FEATURE_COLUMNS)),
        "encoded_feature_count": int(len(feature_names)),
        "duplicate_rows_audit": int(raw_df.duplicated().sum()),
        "dummy_baseline": {
            "accuracy": dummy_metrics["accuracy"],
            "precision": dummy_metrics["precision"],
            "recall": dummy_metrics["recall"],
            "f1_score": dummy_metrics["f1_score"],
            "roc_auc": dummy_metrics["roc_auc"],
            "pr_auc": dummy_metrics["pr_auc"],
        },
    }
)

save_metrics(final_metrics, METRICS_PATH)
print(f"บันทึกผลการประเมินแล้ว: {METRICS_PATH.relative_to(PROJECT_ROOT)}")

## สรุป

การสรุปว่าโมเดลใดดีกว่าจะทำหลังจาก SVM พร้อมแล้ว โดยใช้ test set เดียวกันและพิจารณาตามลำดับ:

1. F1-score ของคลาส Died
2. PR-AUC
3. Recall ของคลาส Died
4. Precision
5. Accuracy ใช้ประกอบเท่านั้น

ขั้นตอนถัดไปคือเปิด Pull Request ของ Decision Tree และรอผลจาก `notebooks/svm.ipynb` ก่อนสร้าง `model_comparison.ipynb`